In [7]:
#LINES Working

import re
import pdfplumber
from datetime import datetime, timedelta


#DOWS = {"Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"}


def parse_bid_range(page_text):
    m = re.search(
        r"Bid Period Date Range:\s*(\d{1,2}[A-Za-z]{3}\d{4})\s*-\s*(\d{1,2}[A-Za-z]{3}\d{4})",
        page_text,
    )
    if not m:
        raise ValueError("Could not find Bid Period Date Range")

    start = datetime.strptime(m.group(1), "%d%b%Y").date()
    end = datetime.strptime(m.group(2), "%d%b%Y").date()
    return start, end


def parse_domicile(page_text):
    m = re.search(r"Domicile:\s*([A-Z]{3})", page_text)
    if not m:
        raise ValueError("Could not find domicile")
    return m.group(1)


def words_on_same_line(words, top, tolerance=2.0):
    return [w for w in words if abs(w["top"] - top) <= tolerance]

def build_package_metadata(bid_start, bid_end):
    return {
        "bid_period_date_range": {
            "start": bid_start.isoformat(),
            "end": bid_end.isoformat(),
        },
        "pay_period_date_ranges": {
            "PP1": {
                "start": bid_start.isoformat(),
                "end": (bid_start + timedelta(days=27)).isoformat(),
            },
            "PP2": {
                "start": (bid_start + timedelta(days=28)).isoformat(),
                "end": (bid_start + timedelta(days=55)).isoformat(),
            },
        },
    }

PP_TOP_FROM_CT_OFFSET = 9.55


def find_pp_anchors(block_words):
    """
    Finds PP1 / PP2 sections using the CT: rows instead of the PP1/PP2 labels.

    This is more reliable because in VTO/VOR/RA/etc. pages,
    pdfplumber can merge hidden assignment text with the PP label.
    """
    ct_words = [
        w for w in block_words
        if w["text"] == "CT:"
        and 35 <= w["x0"] <= 100
    ]

    ct_words = sorted(ct_words, key=lambda w: w["top"])

    pp_anchors = []

    for i, ct_word in enumerate(ct_words[:2], start=1):
        pp_anchors.append({
            "pp_index": i,
            "top": ct_word["top"] - PP_TOP_FROM_CT_OFFSET,
        })

    return pp_anchors

def find_line_blocks(words, domicile):
    """
    Finds line blocks such as:
        SDF 1
        SDF 17
        SDF 18

    Returns y-ranges for each line block.
    """
    starts = []

    for w in words:
        if w["text"] == domicile and w["x0"] < 80 and w["top"] > 120:
            same_line = words_on_same_line(words, w["top"], tolerance=1.5)

            possible_numbers = [
                x for x in same_line
                if x["x0"] > w["x1"]
                and x["x0"] < 100
                and re.fullmatch(r"\d+", x["text"])
            ]

            if possible_numbers:
                starts.append({
                    "line_number": int(possible_numbers[0]["text"]),
                    "top": w["top"],
                })

    starts = sorted(starts, key=lambda x: x["top"])

    for i, block in enumerate(starts):
        if i + 1 < len(starts):
            block["bottom"] = starts[i + 1]["top"] - 5
        else:
            block["bottom"] = 99999

    return starts


def get_metric(words, pp_top, label):
    """
    Gets CT, BT, DO, DD from the left side of each PP row.
    """
    for w in words:
        if (
            w["text"] == label
            and w["x0"] < 90
            and pp_top <= w["top"] <= pp_top + 45
        ):
            same_line = sorted(
                words_on_same_line(words, w["top"], tolerance=1.5),
                key=lambda x: x["x0"],
            )

            after_label = [x for x in same_line if x["x0"] > w["x1"]]
            if after_label:
                return after_label[0]["text"]

    return None


def get_date_columns(words, pp_top):
    """
    The date numbers are on the line just above the PP label.
    We take only the first 28 date columns.

    Example:
        PP1: 21, 22, 23 ... 17
        PP2: 18, 19, 20 ... 15

    We ignore the extra '-- Mon 19' or '-- Mon 17' text after the 28-day grid.
    """
    date_words = [
        w for w in words
        if pp_top - 15 <= w["top"] <= pp_top - 5
        and re.fullmatch(r"\d{1,2}", w["text"])
        and w["x0"] > 80
    ]

    date_words = sorted(date_words, key=lambda w: w["x0"])
    return date_words[:28]

"""
def get_weekday_columns(words, pp_top):
    weekday_words = [
        w for w in words
        if pp_top - 28 <= w["top"] <= pp_top - 17
        and w["text"] in DOWS
        and w["x0"] > 80
    ]

    weekday_words = sorted(weekday_words, key=lambda w: w["x0"])
    return weekday_words[:28]
"""

def nearest_column_index(x, columns, max_distance=12.5):
    best_idx = None
    best_distance = None

    for i, col in enumerate(columns):
        distance = abs(col["center"] - x)

        if best_distance is None or distance < best_distance:
            best_idx = i
            best_distance = distance

    if best_distance is not None and best_distance <= max_distance:
        return best_idx

    return None


ASSIGNMENT_PATTERN = r"(?:\d+|VTO|VOR|RA|SA|RB|SB)"
TIME_PATTERN = r"(?:[01]\d|2[0-3])[0-5]\d"


TIME_PATTERN = r"(?:[01]\d|2[0-3])[0-5]\d"
ASSIGNMENT_PATTERN = r"(?:\d+|VTO|VOR|RA|SA|RB|SB)"


def word_center(w):
    return (w["x0"] + w["x1"]) / 2


def format_hhmm(token):
    return f"{token[:2]}:{token[2:]}"


def is_valid_time_token(token):
    if not re.fullmatch(r"\d{4}", token):
        return False

    hh = int(token[:2])
    mm = int(token[2:])

    return 0 <= hh <= 23 and 0 <= mm <= 59


def time_minutes(token):
    return int(token[:2]) * 60 + int(token[2:])

def nearest_date_boundary(x, columns):
    """
    Finds the nearest boundary between two adjacent date columns.

    Returns:
        {
            "left_index": i,
            "right_index": i + 1,
            "boundary_x": ...,
            "distance": ...
        }
    """

    best = None

    for i in range(len(columns) - 1):
        left_center = columns[i]["center"]
        right_center = columns[i + 1]["center"]

        boundary_x = (left_center + right_center) / 2
        distance = abs(x - boundary_x)

        if best is None or distance < best["distance"]:
            best = {
                "left_index": i,
                "right_index": i + 1,
                "boundary_x": boundary_x,
                "distance": distance,
            }

    return best 

def choose_trip_column_by_time(
    x,
    start_time_token,
    columns,
    fallback_index=None,
    boundary_tolerance=8,
    noon_cutoff_minutes=12 * 60,
):
    """
    Chooses the correct date column for a trip.

    Boundary case:
        If the trip number is printed near the boundary between two dates:
            1200-2359 -> left/previous date
            0000-1159 -> right/next date

    Normal case:
        Use nearest date column.

    Safety:
        If nearest_column_index() returns None, fall back to the original
        assignment column so the parser does not crash.
    """

    boundary = nearest_date_boundary(x, columns)

    if boundary is not None and boundary["distance"] <= boundary_tolerance:
        if start_time_token is not None and is_valid_time_token(start_time_token):
            if time_minutes(start_time_token) >= noon_cutoff_minutes:
                return boundary["left_index"]
            else:
                return boundary["right_index"]

    nearest_idx = nearest_column_index(x, columns)

    if nearest_idx is not None:
        return nearest_idx

    if fallback_index is not None:
        return fallback_index

    return None


def find_assignment_words(words, target_top, columns, y_tolerance=3):
    assignments = []

    min_center = min(col["center"] for col in columns) - 15
    max_center = max(col["center"] for col in columns) + 15

    for w in words:
        token = w["text"].strip().upper()

        if not re.fullmatch(ASSIGNMENT_PATTERN, token):
            continue

        if abs(w["top"] - target_top) > y_tolerance:
            continue

        x = word_center(w)

        if x < min_center or x > max_center:
            continue

        idx = nearest_column_index(x, columns)

        if idx is None:
            continue

        assignments.append({
            "token": token,
            "word": w,
            "column_index": idx,
        })

    assignments.sort(
        key=lambda item: (
            item["word"]["x0"],
            item["word"]["top"],
        )
    )

    return assignments


def find_start_time_for_trip(words, trip_word, x_tolerance=30, y_window=60):
    """
    Finds the start time associated with a numeric trip ID.

    Visual stack usually looks like:

        310
        RDU RDU RDU
        2310
        33:16

    So we prefer a valid HHMM time below the trip number.
    """

    trip_x = word_center(trip_word)

    candidates = []

    for w in words:
        token = w["text"].strip()

        if not is_valid_time_token(token):
            continue

        x_distance = abs(word_center(w) - trip_x)
        y_distance = abs(w["top"] - trip_word["top"])

        if x_distance > x_tolerance:
            continue

        if y_distance > y_window:
            continue

        is_below_trip = w["top"] > trip_word["top"]

        candidates.append({
            "word": w,
            "token": token,
            "x_distance": x_distance,
            "y_distance": y_distance,
            "is_below_trip": is_below_trip,
        })

    if not candidates:
        return None

    candidates.sort(
        key=lambda c: (
            0 if c["is_below_trip"] else 1,
            c["y_distance"],
            c["x_distance"],
        )
    )

    return candidates[0]

def parse_pp(words, pp_anchor, bid_start):
    pp_index = pp_anchor["pp_index"]
    pp_top = pp_anchor["top"]

    date_words = get_date_columns(words, pp_top)

    if len(date_words) < 28:
        raise ValueError(f"Only found {len(date_words)} date columns for PP{pp_index}")

    pp_start = bid_start + timedelta(days=28 * (pp_index - 1))

    columns = []

    for i, date_word in enumerate(date_words):
        actual_date = pp_start + timedelta(days=i)

        columns.append({
            "index": i,
            "date": actual_date.isoformat(),
            "center": word_center(date_word),
        })

    assignment_words = find_assignment_words(
        words=words,
        target_top=pp_top,
        columns=columns,
    )

    assignments = []

    for item in assignment_words:
        token = item["token"]
        assignment_word = item["word"]

        if token.isdigit():
            start_time_info = find_start_time_for_trip(
                words=words,
                trip_word=assignment_word,
            )

            start_time = None
            date_column_index = item["column_index"]

            if start_time_info is not None:
                start_time_token = start_time_info["token"]
                start_time_word = start_time_info["word"]

                start_time = format_hhmm(start_time_token)

                # Important:
                # Use the time to choose left/right date only when near a date boundary.
                date_column_index = choose_trip_column_by_time(
                            x=word_center(assignment_word),
                            start_time_token=start_time_token,
                            columns=columns,
                            fallback_index=item["column_index"],
                            boundary_tolerance=8,
                            )

            assignments.append({
                "date": columns[date_column_index]["date"],
                "start_time": start_time,
                "type": "trip",
                "value": int(token),
            })

        else:
            # VTO / VOR / RA / SA / RB / SB do not need time logic.
            assignments.append({
                "date": columns[item["column_index"]]["date"],
                "type": "code",
                "value": token,
            })

    return {
        "pp": f"PP{pp_index}",
        "CT": get_metric(words, pp_top, "CT:"),
        "BT": get_metric(words, pp_top, "BT:"),
        "DO": get_metric(words, pp_top, "DO:"),
        "DD": get_metric(words, pp_top, "DD:"),
        "assignments": assignments,
    }
    
def parse_line_report_page(page, bid_start, domicile):
    words = page.extract_words(
        x_tolerance=1,
        y_tolerance=2,
        keep_blank_chars=False
    )

    line_blocks = find_line_blocks(words, domicile)

    parsed_lines = []

    for line_block in line_blocks:
        block_words = [
            w for w in words
            if line_block["top"] - 5 <= w["top"] < line_block["bottom"]
        ]

        pp_anchors = find_pp_anchors(block_words)

        pp_data = []

        for pp_anchor in pp_anchors:
            pp_data.append(parse_pp(block_words, pp_anchor, bid_start))

        parsed_lines.append({
            "line_number": line_block["line_number"],
            "pay_periods": pp_data,
        })

    return parsed_lines


def parse_line_report_pdf(pdf_path, first_calendar_page=3):
    """
    first_calendar_page uses normal PDF page numbering.

    Example:
        first_calendar_page=5 means:
        skip pages 1-4, start extracting line calendar data on page 5.
    """

    first_calendar_index = first_calendar_page - 1

    with pdfplumber.open(pdf_path) as pdf:
        if first_calendar_index >= len(pdf.pages):
            raise ValueError(
                f"first_calendar_page={first_calendar_page} is beyond the end of the PDF. "
                f"The PDF only has {len(pdf.pages)} pages."
            )

        # Read metadata from the first actual calendar page, not PDF page 1.
        metadata_text = pdf.pages[first_calendar_index].extract_text(
            x_tolerance=1,
            y_tolerance=2
        ) or ""

        bid_start, bid_end = parse_bid_range(metadata_text)
        domicile = parse_domicile(metadata_text)

        result = build_package_metadata(bid_start, bid_end)
        result["lines"] = []

        for page in pdf.pages[first_calendar_index:]:
            page_lines = parse_line_report_page(
                page=page,
                bid_start=bid_start,
                domicile=domicile,
            )
            result["lines"].extend(page_lines)

    return result

In [8]:
from pprint import pprint

pdf_path = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/BID2605 lines 757 SDF.pdf"
#pdf_path = "A:/Github Repos/Personal Projects/UPS-project/SamplePDFs/2507 Lines.pdf"

lines = parse_line_report_pdf(pdf_path, first_calendar_page=3)

pprint(lines, width=160)

{'bid_period_date_range': {'end': '2026-09-06', 'start': '2026-07-12'},
 'lines': [{'line_number': 1,
            'pay_periods': [{'BT': '45:10',
                             'CT': '72:12',
                             'DD': '12',
                             'DO': '12',
                             'assignments': [{'date': '2026-07-14', 'start_time': '05:00', 'type': 'trip', 'value': 428},
                                             {'date': '2026-07-15', 'start_time': '05:00', 'type': 'trip', 'value': 428},
                                             {'date': '2026-07-16', 'start_time': '06:57', 'type': 'trip', 'value': 516},
                                             {'date': '2026-07-17', 'start_time': '06:57', 'type': 'trip', 'value': 516},
                                             {'date': '2026-07-21', 'start_time': '05:00', 'type': 'trip', 'value': 428},
                                             {'date': '2026-07-22', 'start_time': '05:00', 'type': 'trip', 'value': 42

In [ ]:
#TRIPS Working 1
import re
import pdfplumber


TIME_RE = r"\(\d{2}\)\d{2}:\d{2}"
DUR_RE = r"\d+h\d{2}"

def clean_time(value):
    """
    Converts:
        (16)20:43 -> 20:43
        (00)04:09 -> 04:09
    """
    if value is None:
        return None

    return re.sub(r"^\(\d{2}\)", "", value)


def group_words_by_line(words, tolerance=2):
    lines = []

    for word in sorted(words, key=lambda w: (w["top"], w["x0"])):
        for line in lines:
            if abs(line[0]["top"] - word["top"]) <= tolerance:
                line.append(word)
                break
        else:
            lines.append([word])

    return [sorted(line, key=lambda w: w["x0"]) for line in lines]


def find_trip_anchors(page):
    """
    Finds each 'Trip Id: ###' on the page.
    Needed because each page can contain several trip tables.
    """
    words = page.extract_words(x_tolerance=1, y_tolerance=3) or []
    anchors = []

    for line in group_words_by_line(words):
        text_parts = [w["text"] for w in line]

        for i in range(len(text_parts) - 2):
            if (
                text_parts[i] == "Trip"
                and text_parts[i + 1] == "Id:"
                and text_parts[i + 2].isdigit()
            ):
                anchors.append({
                    "trip_id": int(text_parts[i + 2]),
                    "x0": line[i]["x0"],
                    "top": line[i]["top"],
                })

    return sorted(anchors, key=lambda a: (a["top"], a["x0"]))


def make_trip_crops(page):
    """
    Builds a crop box for each trip table.

    The crop is necessary because the PDF pages can have left/right tables.
    Without cropping, pdfplumber may mix text from different tables.
    """
    anchors = find_trip_anchors(page)
    page_middle = page.width / 2

    for anchor in anchors:
        anchor["column"] = 0 if anchor["x0"] < page_middle else 1

    crops = []

    for anchor in anchors:
        next_trip_same_column = [
            other
            for other in anchors
            if other["column"] == anchor["column"]
            and other["top"] > anchor["top"] + 5
        ]

        bottom = min(
            [other["top"] for other in next_trip_same_column],
            default=page.height - 10,
        )

        if anchor["column"] == 0:
            x0, x1 = 0, page_middle - 2
        else:
            x0, x1 = page_middle + 2, page.width

        crops.append((
            x0,
            max(0, anchor["top"] - 3),
            x1,
            min(page.height, bottom - 2),
        ))

    return crops


def split_route(route_raw):
    """
    Examples:
        SDF-PHL         -> departure SDF, arrival PHL, route_flags []
        SDF-IRO-PHL     -> departure SDF, arrival PHL, route_flags ['IRO']
        SDF-BDL(C)      -> departure SDF, arrival BDL, route_flags ['C']
        SDF-IRO-BDL(C)  -> departure SDF, arrival BDL, route_flags ['IRO', 'C']
    """
    parts = route_raw.split("-")

    airports = []
    route_flags = []

    for part in parts:
        if part == "IRO":
            route_flags.append("IRO")
            continue

        # Handles airport with parenthetical flag, like BDL(C)
        match = re.match(r"^([A-Z]{3})(?:\(([A-Z]+)\))?$", part)

        if match:
            airport = match.group(1)
            flag = match.group(2)

            airports.append(airport)

            if flag:
                route_flags.append(flag)
        else:
            airports.append(part)

    departure = airports[0] if airports else None
    arrival = airports[-1] if airports else None

    return departure, arrival, route_flags


def parse_flight_line(line):
    """
    Extracts only:
    - flight
    - route_raw
    - departure
    - arrival
    - route_flags, such as IRO
    - start
    - end
    """

    match = re.match(r"^\d+\s+\([^)]*\)[A-Za-z]{0,2}\s+(.*)$", line)

    if not match:
        return None

    body = match.group(1)

    route_match = re.search(
        r"[A-Z]{3}(?:\([A-Z]\))?(?:-(?:IRO|[A-Z]{3}(?:\([A-Z]\))?))+",
        body,
    )

    if not route_match:
        return None

    flight = body[:route_match.start()].strip()
    route_raw = route_match.group(0)
    after_route = body[route_match.end():].strip()

    time_match = re.match(
        rf"(?P<start>{TIME_RE})\s+(?P<end>{TIME_RE})",
        after_route,
    )

    if not time_match:
        return None

    departure, arrival, route_flags = split_route(route_raw)

    return {
        "flight": flight,
        "route_raw": route_raw,
        "departure": departure,
        "arrival": arrival,
        "route_flags": route_flags,
        "start": clean_time(time_match.group("start")),
        "end": clean_time(time_match.group("end")),
    }


def parse_trip_text(text):
    """
    Parses one trip table into a dictionary.
    """

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    full_text = "\n".join(lines)

    trip_id = int(re.search(r"Trip Id:\s*(\d+)", full_text).group(1))

    lines_match = re.search(r"Lines:\s*([^\n]+)", full_text)
    line_numbers = []

    if lines_match:
        line_numbers = [int(x) for x in re.findall(r"\d+", lines_match.group(1))]

    tafb_match = re.search(r"TAFB:\s*(\d+h\d{2})", full_text)
    premium_match = re.search(r"Premium\s+([\d.]+)", full_text)

    trip = {
        "trip_id": trip_id,
        "lines": line_numbers,
        "total_blocks": 0,
        "tafb": tafb_match.group(1) if tafb_match else None,
        "premium": float(premium_match.group(1)) if premium_match else None,
        "blocks": [],
    }

    current_block = None

    for line in lines:
        # New block starts with something like:
        # (15)19:43 1h00 Duty 8h26
        duty_match = re.match(
            rf"^(?P<block_start>{TIME_RE})\s+{DUR_RE}\s+Duty\s+{DUR_RE}",
            line,
        )

        if duty_match:
            current_block = {
                "start": clean_time(duty_match.group("block_start")),
                "end": None,
                "rest": None,
                "flights": [],
            }
            trip["blocks"].append(current_block)
            continue

        if current_block is None:
            continue

        # Block ends with something like:
        # (00)04:09 0h15 Credit 4h13D
        block_end_match = re.match(
            rf"^(?P<block_end>{TIME_RE})\s+"
            rf"(?P<ground_time>{DUR_RE})"
            rf"(?:\s+(?P<after>.*))?$",
            line,
        )

        if block_end_match:
            after = block_end_match.group("after") or ""

            # If the word immediately after the duration is Duty,
            # this is a new block start, not a block end.
            if not after.startswith("Duty"):
                current_block["end"] = clean_time(block_end_match.group("block_end"))

        flight = parse_flight_line(line)

        if flight:
            # For the first flight in each block, use the block/duty start time,
            # not the actual flight departure time.
            if not current_block["flights"]:
                flight["start"] = current_block["start"]

            current_block["flights"].append(flight)

        rest_match = re.search(r"Rest\s+(-|\d+h\d{2})", line)

        if rest_match:
            current_block["rest"] = rest_match.group(1)

    trip["total_blocks"] = len(trip["blocks"])

    return trip


def extract_trips_from_pdf(pdf_path, first_page=2, last_page=None):
    """
    Returns a dictionary keyed by Trip ID.

    Page numbers are normal PDF page numbers:
        first_page=2 means page 2.
        last_page=4 means page 4.
    """

    trips = {}

    with pdfplumber.open(pdf_path) as pdf:
        start_index = first_page - 1
        end_index = last_page if last_page is not None else len(pdf.pages)

        for page_index in range(start_index, end_index):
            page = pdf.pages[page_index]

            for bbox in make_trip_crops(page):
                cropped = page.crop(bbox)
                text = cropped.extract_text(x_tolerance=1, y_tolerance=3) or ""

                if "Trip Id:" not in text:
                    continue

                trip = parse_trip_text(text)
                trips[trip["trip_id"]] = trip

    return trips

In [ ]:
from pprint import pprint

pdf_path = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Trips.pdf"

trips = extract_trips_from_pdf(pdf_path,first_page=2)

pprint(trips[607])

In [4]:
#TRIPS working Progress report + EXTRA INFO
import re
import pdfplumber


TIME_RE = r"\(\d{2}\)\d{2}:\d{2}"
DUR_RE = r"\d+h\d{2}"

def get_first_match(pattern, text, default=None):
    match = re.search(pattern, text)
    return match.group(1) if match else default


def get_first_float(pattern, text, default=None):
    value = get_first_match(pattern, text, default=None)
    return float(value) if value is not None else default


def get_first_int(pattern, text, default=None):
    value = get_first_match(pattern, text, default=None)
    return int(value) if value is not None else default

def clean_time(value):
    """
    Converts:
        (16)20:43 -> 20:43
        (00)04:09 -> 04:09
    """
    if value is None:
        return None

    return re.sub(r"^\(\d{2}\)", "", value)


def group_words_by_line(words, tolerance=2):
    lines = []

    for word in sorted(words, key=lambda w: (w["top"], w["x0"])):
        for line in lines:
            if abs(line[0]["top"] - word["top"]) <= tolerance:
                line.append(word)
                break
        else:
            lines.append([word])

    return [sorted(line, key=lambda w: w["x0"]) for line in lines]


def find_trip_anchors(page):
    """
    Finds each 'Trip Id: ###' on the page.
    Needed because each page can contain several trip tables.
    """
    words = page.extract_words(x_tolerance=1, y_tolerance=3) or []
    anchors = []

    for line in group_words_by_line(words):
        text_parts = [w["text"] for w in line]

        for i in range(len(text_parts) - 2):
            if (
                text_parts[i] == "Trip"
                and text_parts[i + 1] == "Id:"
                and text_parts[i + 2].isdigit()
            ):
                anchors.append({
                    "trip_id": int(text_parts[i + 2]),
                    "x0": line[i]["x0"],
                    "top": line[i]["top"],
                })

    return sorted(anchors, key=lambda a: (a["top"], a["x0"]))


def make_trip_crops(page):
    """
    Builds a crop box for each trip table.

    The crop is necessary because the PDF pages can have left/right tables.
    Without cropping, pdfplumber may mix text from different tables.
    """
    anchors = find_trip_anchors(page)
    page_middle = page.width / 2

    for anchor in anchors:
        anchor["column"] = 0 if anchor["x0"] < page_middle else 1

    crops = []

    for anchor in anchors:
        next_trip_same_column = [
            other
            for other in anchors
            if other["column"] == anchor["column"]
            and other["top"] > anchor["top"] + 5
        ]

        bottom = min(
            [other["top"] for other in next_trip_same_column],
            default=page.height - 10,
        )

        if anchor["column"] == 0:
            x0, x1 = 0, page_middle - 2
        else:
            x0, x1 = page_middle + 2, page.width

        crops.append((
            x0,
            max(0, anchor["top"] - 3),
            x1,
            min(page.height, bottom - 2),
        ))

    return crops


def split_route(route_raw):
    """
    Examples:
        SDF-PHL         -> departure SDF, arrival PHL, route_flags []
        SDF-IRO-PHL     -> departure SDF, arrival PHL, route_flags ['IRO']
        SDF-BDL(C)      -> departure SDF, arrival BDL, route_flags ['C']
        SDF-IRO-BDL(C)  -> departure SDF, arrival BDL, route_flags ['IRO', 'C']
    """
    parts = route_raw.split("-")

    airports = []
    route_flags = []

    for part in parts:
        if part == "IRO":
            route_flags.append("IRO")
            continue

        # Handles airport with parenthetical flag, like BDL(C)
        match = re.match(r"^([A-Z]{3})(?:\(([A-Z]+)\))?$", part)

        if match:
            airport = match.group(1)
            flag = match.group(2)

            airports.append(airport)

            if flag:
                route_flags.append(flag)
        else:
            airports.append(part)

    departure = airports[0] if airports else None
    arrival = airports[-1] if airports else None

    return departure, arrival, route_flags


def parse_flight_line(line):
    """
    Extracts only:
    - flight
    - route_raw
    - departure
    - arrival
    - route_flags, such as IRO
    - start
    - end
    """

    match = re.match(r"^\d+\s+\([^)]*\)[A-Za-z]{0,2}\s+(.*)$", line)

    if not match:
        return None

    body = match.group(1)

    route_match = re.search(
        r"[A-Z]{3}(?:\([A-Z]\))?(?:-(?:IRO|[A-Z]{3}(?:\([A-Z]\))?))+",
        body,
    )

    if not route_match:
        return None

    flight = body[:route_match.start()].strip()
    route_raw = route_match.group(0)
    after_route = body[route_match.end():].strip()

    time_match = re.match(
        rf"(?P<start>{TIME_RE})\s+(?P<end>{TIME_RE})",
        after_route,
    )

    if not time_match:
        return None

    departure, arrival, route_flags = split_route(route_raw)

    return {
        "flight": flight,
        "route_raw": route_raw,
        "departure": departure,
        "arrival": arrival,
        "route_flags": route_flags,
        "start": clean_time(time_match.group("start")),
        "end": clean_time(time_match.group("end")),
    }


def parse_trip_text(text):
    """
    Parses one trip table into a dictionary.

    Extracts:
        Trip-level:
            - trip_id
            - lines
            - total_blocks
            - tafb
            - premium
            - duty_time
            - block_time
            - credit_time
            - per_diem
            - ldgs

        Block-level:
            - start
            - end
            - duty
            - block
            - rest
            - credit
            - flights
    """

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    full_text = "\n".join(lines)

    trip_id = int(re.search(r"Trip Id:\s*(\d+)", full_text).group(1))

    lines_match = re.search(r"Lines:\s*([^\n]+)", full_text)
    line_numbers = []

    if lines_match:
        line_numbers = [int(x) for x in re.findall(r"\d+", lines_match.group(1))]

    trip = {
        "trip_id": trip_id,
        "lines": line_numbers,
        "total_blocks": 0,

        # Trip summary values
        "tafb": get_first_match(r"TAFB:\s*(\d+h\d{2})", full_text),
        "premium": get_first_float(r"Premium\s+([\d.]+)", full_text),
        "duty_time": get_first_match(r"Duty Time:\s*(\d+h\d{2})", full_text),
        "block_time": get_first_match(r"Block Time:\s*(\d+h\d{2})", full_text),
        "credit_time": get_first_match(r"Credit Time:\s*(\d+h\d{2}[A-Z]?)", full_text),
        "per_diem": get_first_float(r"per Diem\s+([\d.]+)", full_text),
        "ldgs": get_first_int(r"LDGS\s+(\d+)", full_text),

        # Block list
        "blocks": [],
    }

    current_block = None

    for line in lines:
        # New block starts with something like:
        # (15)19:43 1h00 Duty 8h26
        # (16)21:00 0h00 Duty 0h00
        duty_match = re.match(
            rf"^(?P<block_start>{TIME_RE})\s+"
            rf"(?P<report_time>{DUR_RE})\s+"
            rf"Duty\s+(?P<duty>{DUR_RE})",
            line,
        )

        if duty_match:
            current_block = {
                "start": clean_time(duty_match.group("block_start")),
                "end": None,

                # New block-level fields
                "duty": duty_match.group("duty"),
                "block": None,
                "rest": None,
                "credit": None,

                "flights": [],
            }
            trip["blocks"].append(current_block)
            continue

        if current_block is None:
            continue

        # Block ends can look like:
        # (00)04:09 0h15 Credit 4h13D
        # (03)07:00 0h00 Rest -
        # (20)00:51 0h15 Credit 4h00M
        #
        # But this should NOT count as a block end:
        # (16)21:00 0h00 Duty 0h00
        block_end_match = re.match(
            rf"^(?P<block_end>{TIME_RE})\s+"
            rf"(?P<ground_time>{DUR_RE})"
            rf"(?:\s+(?P<after>.*))?$",
            line,
        )

        if block_end_match:
            after = block_end_match.group("after") or ""

            if not after.startswith("Duty"):
                current_block["end"] = clean_time(block_end_match.group("block_end"))

        flight = parse_flight_line(line)

        if flight:
            # For the first flight in each block, use the duty/block start time,
            # not the actual flight departure time.
            if not current_block["flights"]:
                flight["start"] = current_block["start"]

            current_block["flights"].append(flight)

        # Block-level Block.
        # Important: this should match "Block 3h19",
        # but NOT "Block Time: 3h19".
        block_match = re.search(
            r"\bBlock\s+(?!Time:)(\d+h\d{2})",
            line,
        )

        if block_match:
            current_block["block"] = block_match.group(1)

        # Block-level Rest.
        rest_match = re.search(
            r"\bRest\s+(-|\d+h\d{2})",
            line,
        )

        if rest_match:
            current_block["rest"] = rest_match.group(1)

        # Block-level Credit.
        # Important: this should match "Credit 5h30M",
        # but NOT "Credit Time: 5h30D".
        credit_match = re.search(
            r"\bCredit\s+(?!Time:)(\d+h\d{2}[A-Z]?)",
            line,
        )

        if credit_match:
            current_block["credit"] = credit_match.group(1)

    trip["total_blocks"] = len(trip["blocks"])

    return trip


import pdfplumber


def extract_trips_from_pdf(
    pdf_path,
    first_page=2,
    last_page=None,
    stop_after_empty_pages=4,
    progress_callback=None,
):
    """
    Returns a dictionary keyed by Trip ID.

    progress_callback:
        Optional function that receives a progress dictionary.

    Example progress data:
        {
            "current": 10,
            "total": 150,
            "page": 11,
            "trips_on_page": 8,
            "total_trips": 72,
            "status": "running",
            "message": "Extracting page 11 of 150",
        }
    """

    def send_progress(
        current,
        total,
        page_number=None,
        trips_on_page=0,
        total_trips=0,
        status="running",
        message=None,
    ):
        if progress_callback is None:
            return

        progress_callback({
            "current": current,
            "total": total,
            "page": page_number,
            "trips_on_page": trips_on_page,
            "total_trips": total_trips,
            "status": status,
            "message": message,
        })

    trips = {}
    empty_pages_in_a_row = 0

    with pdfplumber.open(pdf_path) as pdf:
        total_pdf_pages = len(pdf.pages)

        start_index = first_page - 1
        end_index = last_page if last_page is not None else total_pdf_pages

        start_index = max(0, start_index)
        end_index = min(end_index, total_pdf_pages)

        total_pages_to_process = end_index - start_index

        if total_pages_to_process <= 0:
            send_progress(
                current=0,
                total=0,
                status="done",
                message="No pages to process.",
            )
            return trips

        send_progress(
            current=0,
            total=total_pages_to_process,
            status="starting",
            message="Starting trip extraction...",
        )

        for page_index in range(start_index, end_index):
            page_number = page_index + 1
            current_progress = page_index - start_index + 1

            page = pdf.pages[page_index]
            trips_found_on_page = 0

            try:
                for bbox in make_trip_crops(page):
                    cropped = page.crop(bbox)
                    text = cropped.extract_text(x_tolerance=1, y_tolerance=3) or ""

                    if "Trip Id:" not in text:
                        continue

                    trip = parse_trip_text(text)
                    trips[trip["trip_id"]] = trip
                    trips_found_on_page += 1

            finally:
                page.close()

            if trips_found_on_page == 0:
                empty_pages_in_a_row += 1
            else:
                empty_pages_in_a_row = 0

            send_progress(
                current=current_progress,
                total=total_pages_to_process,
                page_number=page_number,
                trips_on_page=trips_found_on_page,
                total_trips=len(trips),
                status="running",
                message=(
                    f"Extracting trips: page {page_number} "
                    f"({current_progress} of {total_pages_to_process})"
                ),
            )

            if (
                stop_after_empty_pages is not None
                and empty_pages_in_a_row >= stop_after_empty_pages
            ):
                send_progress(
                    current=current_progress,
                    total=total_pages_to_process,
                    page_number=page_number,
                    trips_on_page=trips_found_on_page,
                    total_trips=len(trips),
                    status="stopped",
                    message=(
                        f"Stopped at page {page_number}: "
                        f"{empty_pages_in_a_row} empty pages in a row."
                    ),
                )
                break

        send_progress(
            current=min(current_progress, total_pages_to_process),
            total=total_pages_to_process,
            page_number=page_number,
            total_trips=len(trips),
            status="done",
            message=f"Finished extracting {len(trips)} trips.",
        )

    return trips

In [5]:
pdf_path = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Trips.pdf"
def print_progress(progress):
    current = progress["current"]
    total = progress["total"]
    status = progress["status"]
    message = progress["message"]

    if total:
        percent = current / total * 100
        print(f"{percent:5.1f}% - {message}")
    else:
        print(f"{status}: {message}")


trips = extract_trips_from_pdf(
    pdf_path,
    first_page=2,
    stop_after_empty_pages=8,
    progress_callback=print_progress,
)

  0.0% - Starting trip extraction...
  0.7% - Extracting trips: page 2 (1 of 150)
  1.3% - Extracting trips: page 3 (2 of 150)
  2.0% - Extracting trips: page 4 (3 of 150)
  2.7% - Extracting trips: page 5 (4 of 150)
  3.3% - Extracting trips: page 6 (5 of 150)
  4.0% - Extracting trips: page 7 (6 of 150)
  4.7% - Extracting trips: page 8 (7 of 150)
  5.3% - Extracting trips: page 9 (8 of 150)
  6.0% - Extracting trips: page 10 (9 of 150)
  6.7% - Extracting trips: page 11 (10 of 150)
  7.3% - Extracting trips: page 12 (11 of 150)
  8.0% - Extracting trips: page 13 (12 of 150)
  8.7% - Extracting trips: page 14 (13 of 150)
  9.3% - Extracting trips: page 15 (14 of 150)
 10.0% - Extracting trips: page 16 (15 of 150)
 10.7% - Extracting trips: page 17 (16 of 150)
 11.3% - Extracting trips: page 18 (17 of 150)
 12.0% - Extracting trips: page 19 (18 of 150)
 12.7% - Extracting trips: page 20 (19 of 150)
 13.3% - Extracting trips: page 21 (20 of 150)
 14.0% - Extracting trips: page 22 (21 o

In [6]:
from pprint import pprint
pprint(trips)

{1: {'block_time': '3h42',
     'blocks': [{'block': '3h42',
                 'credit': '5h18D',
                 'duty': '7h57',
                 'end': '14:10',
                 'flights': [{'arrival': 'JFK',
                              'departure': 'SDF',
                              'end': '08:55',
                              'flight': '1114',
                              'route_flags': [],
                              'route_raw': 'SDF-JFK',
                              'start': '06:13'},
                             {'arrival': 'SDF',
                              'departure': 'JFK',
                              'end': '13:55',
                              'flight': '2111',
                              'route_flags': [],
                              'route_raw': 'JFK-SDF',
                              'start': '11:55'}],
                 'rest': '-',
                 'start': '06:13'}],
     'credit_time': '6h00M',
     'duty_time': '7h57',
     'ldgs': 2,
     'line

In [ ]:
#TRIPS Testing faster version no Cropping
import re
import pdfplumber


TIME_RE = r"\(\d{2}\)\d{2}:\d{2}"
DUR_RE = r"\d+h\d{2}"

def find_trip_anchors_from_words(words):
    """
    Same idea as find_trip_anchors(page), but uses words already extracted
    from the page instead of calling page.extract_words() again.
    """
    anchors = []

    for line in group_words_by_line(words):
        text_parts = [w["text"] for w in line]

        for i in range(len(text_parts) - 2):
            if (
                text_parts[i] == "Trip"
                and text_parts[i + 1] == "Id:"
                and text_parts[i + 2].isdigit()
            ):
                anchors.append({
                    "trip_id": int(text_parts[i + 2]),
                    "x0": line[i]["x0"],
                    "top": line[i]["top"],
                })

    return sorted(anchors, key=lambda a: (a["top"], a["x0"]))


def make_trip_crops_from_words(page, words):
    """
    Builds trip crop boxes using words already extracted from the page.
    This avoids calling page.extract_words() inside make_trip_crops().
    """
    anchors = find_trip_anchors_from_words(words)
    page_middle = page.width / 2

    for anchor in anchors:
        anchor["column"] = 0 if anchor["x0"] < page_middle else 1

    crops = []

    for anchor in anchors:
        next_trip_same_column = [
            other
            for other in anchors
            if other["column"] == anchor["column"]
            and other["top"] > anchor["top"] + 5
        ]

        bottom = min(
            [other["top"] for other in next_trip_same_column],
            default=page.height - 10,
        )

        if anchor["column"] == 0:
            x0, x1 = 0, page_middle - 2
        else:
            x0, x1 = page_middle + 2, page.width

        crops.append((
            x0,
            max(0, anchor["top"] - 3),
            x1,
            min(page.height, bottom - 2),
        ))

    return crops


def words_inside_bbox(words, bbox):
    """
    Returns only words that visually fall inside the trip table bbox.
    bbox format: (x0, top, x1, bottom)
    """
    x0, top, x1, bottom = bbox

    return [
        word
        for word in words
        if word["x0"] >= x0
        and word["x1"] <= x1
        and word["top"] >= top
        and word["bottom"] <= bottom
    ]


def words_to_text(words, line_tolerance=2):
    """
    Rebuilds text from already-extracted words.
    This replaces cropped.extract_text().
    """
    text_lines = []

    for line_words in group_words_by_line(words, tolerance=line_tolerance):
        text_lines.append(
            " ".join(word["text"] for word in line_words)
        )

    return "\n".join(text_lines)

def get_first_match(pattern, text, default=None):
    match = re.search(pattern, text)
    return match.group(1) if match else default


def get_first_float(pattern, text, default=None):
    value = get_first_match(pattern, text, default=None)
    return float(value) if value is not None else default


def get_first_int(pattern, text, default=None):
    value = get_first_match(pattern, text, default=None)
    return int(value) if value is not None else default

def clean_time(value):
    """
    Converts:
        (16)20:43 -> 20:43
        (00)04:09 -> 04:09
    """
    if value is None:
        return None

    return re.sub(r"^\(\d{2}\)", "", value)


def group_words_by_line(words, tolerance=2):
    lines = []

    for word in sorted(words, key=lambda w: (w["top"], w["x0"])):
        for line in lines:
            if abs(line[0]["top"] - word["top"]) <= tolerance:
                line.append(word)
                break
        else:
            lines.append([word])

    return [sorted(line, key=lambda w: w["x0"]) for line in lines]


def find_trip_anchors(page):
    """
    Finds each 'Trip Id: ###' on the page.
    Needed because each page can contain several trip tables.
    """
    words = page.extract_words(x_tolerance=1, y_tolerance=3) or []
    anchors = []

    for line in group_words_by_line(words):
        text_parts = [w["text"] for w in line]

        for i in range(len(text_parts) - 2):
            if (
                text_parts[i] == "Trip"
                and text_parts[i + 1] == "Id:"
                and text_parts[i + 2].isdigit()
            ):
                anchors.append({
                    "trip_id": int(text_parts[i + 2]),
                    "x0": line[i]["x0"],
                    "top": line[i]["top"],
                })

    return sorted(anchors, key=lambda a: (a["top"], a["x0"]))


def make_trip_crops(page):
    """
    Builds a crop box for each trip table.

    The crop is necessary because the PDF pages can have left/right tables.
    Without cropping, pdfplumber may mix text from different tables.
    """
    anchors = find_trip_anchors(page)
    page_middle = page.width / 2

    for anchor in anchors:
        anchor["column"] = 0 if anchor["x0"] < page_middle else 1

    crops = []

    for anchor in anchors:
        next_trip_same_column = [
            other
            for other in anchors
            if other["column"] == anchor["column"]
            and other["top"] > anchor["top"] + 5
        ]

        bottom = min(
            [other["top"] for other in next_trip_same_column],
            default=page.height - 10,
        )

        if anchor["column"] == 0:
            x0, x1 = 0, page_middle - 2
        else:
            x0, x1 = page_middle + 2, page.width

        crops.append((
            x0,
            max(0, anchor["top"] - 3),
            x1,
            min(page.height, bottom - 2),
        ))

    return crops


def split_route(route_raw):
    """
    Examples:
        SDF-PHL         -> departure SDF, arrival PHL, route_flags []
        SDF-IRO-PHL     -> departure SDF, arrival PHL, route_flags ['IRO']
        SDF-BDL(C)      -> departure SDF, arrival BDL, route_flags ['C']
        SDF-IRO-BDL(C)  -> departure SDF, arrival BDL, route_flags ['IRO', 'C']
    """
    parts = route_raw.split("-")

    airports = []
    route_flags = []

    for part in parts:
        if part == "IRO":
            route_flags.append("IRO")
            continue

        # Handles airport with parenthetical flag, like BDL(C)
        match = re.match(r"^([A-Z]{3})(?:\(([A-Z]+)\))?$", part)

        if match:
            airport = match.group(1)
            flag = match.group(2)

            airports.append(airport)

            if flag:
                route_flags.append(flag)
        else:
            airports.append(part)

    departure = airports[0] if airports else None
    arrival = airports[-1] if airports else None

    return departure, arrival, route_flags


def parse_flight_line(line):
    """
    Extracts only:
    - flight
    - route_raw
    - departure
    - arrival
    - route_flags, such as IRO
    - start
    - end
    """

    match = re.match(r"^\d+\s+\([^)]*\)[A-Za-z]{0,2}\s+(.*)$", line)

    if not match:
        return None

    body = match.group(1)

    route_match = re.search(
        r"[A-Z]{3}(?:\([A-Z]\))?(?:-(?:IRO|[A-Z]{3}(?:\([A-Z]\))?))+",
        body,
    )

    if not route_match:
        return None

    flight = body[:route_match.start()].strip()
    route_raw = route_match.group(0)
    after_route = body[route_match.end():].strip()

    time_match = re.match(
        rf"(?P<start>{TIME_RE})\s+(?P<end>{TIME_RE})",
        after_route,
    )

    if not time_match:
        return None

    departure, arrival, route_flags = split_route(route_raw)

    return {
        "flight": flight,
        "route_raw": route_raw,
        "departure": departure,
        "arrival": arrival,
        "route_flags": route_flags,
        "start": clean_time(time_match.group("start")),
        "end": clean_time(time_match.group("end")),
    }


def parse_trip_text(text):
    """
    Parses one trip table into a dictionary.

    Extracts:
        Trip-level:
            - trip_id
            - lines
            - total_blocks
            - tafb
            - premium
            - duty_time
            - block_time
            - credit_time
            - per_diem
            - ldgs

        Block-level:
            - start
            - end
            - duty
            - block
            - rest
            - credit
            - flights
    """

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    full_text = "\n".join(lines)

    trip_id = int(re.search(r"Trip Id:\s*(\d+)", full_text).group(1))

    lines_match = re.search(r"Lines:\s*([^\n]+)", full_text)
    line_numbers = []

    if lines_match:
        line_numbers = [int(x) for x in re.findall(r"\d+", lines_match.group(1))]

    trip = {
        "trip_id": trip_id,
        "lines": line_numbers,
        "total_blocks": 0,

        # Trip summary values
        "tafb": get_first_match(r"TAFB:\s*(\d+h\d{2})", full_text),
        "premium": get_first_float(r"Premium\s+([\d.]+)", full_text),
        "duty_time": get_first_match(r"Duty Time:\s*(\d+h\d{2})", full_text),
        "block_time": get_first_match(r"Block Time:\s*(\d+h\d{2})", full_text),
        "credit_time": get_first_match(r"Credit Time:\s*(\d+h\d{2}[A-Z]?)", full_text),
        "per_diem": get_first_float(r"per Diem\s+([\d.]+)", full_text),
        "ldgs": get_first_int(r"LDGS\s+(\d+)", full_text),

        # Block list
        "blocks": [],
    }

    current_block = None

    for line in lines:
        # New block starts with something like:
        # (15)19:43 1h00 Duty 8h26
        # (16)21:00 0h00 Duty 0h00
        duty_match = re.match(
            rf"^(?P<block_start>{TIME_RE})\s+"
            rf"(?P<report_time>{DUR_RE})\s+"
            rf"Duty\s+(?P<duty>{DUR_RE})",
            line,
        )

        if duty_match:
            current_block = {
                "start": clean_time(duty_match.group("block_start")),
                "end": None,

                # New block-level fields
                "duty": duty_match.group("duty"),
                "block": None,
                "rest": None,
                "credit": None,

                "flights": [],
            }
            trip["blocks"].append(current_block)
            continue

        if current_block is None:
            continue

        # Block ends can look like:
        # (00)04:09 0h15 Credit 4h13D
        # (03)07:00 0h00 Rest -
        # (20)00:51 0h15 Credit 4h00M
        #
        # But this should NOT count as a block end:
        # (16)21:00 0h00 Duty 0h00
        block_end_match = re.match(
            rf"^(?P<block_end>{TIME_RE})\s+"
            rf"(?P<ground_time>{DUR_RE})"
            rf"(?:\s+(?P<after>.*))?$",
            line,
        )

        if block_end_match:
            after = block_end_match.group("after") or ""

            if not after.startswith("Duty"):
                current_block["end"] = clean_time(block_end_match.group("block_end"))

        flight = parse_flight_line(line)

        if flight:
            # For the first flight in each block, use the duty/block start time,
            # not the actual flight departure time.
            if not current_block["flights"]:
                flight["start"] = current_block["start"]

            current_block["flights"].append(flight)

        # Block-level Block.
        # Important: this should match "Block 3h19",
        # but NOT "Block Time: 3h19".
        block_match = re.search(
            r"\bBlock\s+(?!Time:)(\d+h\d{2})",
            line,
        )

        if block_match:
            current_block["block"] = block_match.group(1)

        # Block-level Rest.
        rest_match = re.search(
            r"\bRest\s+(-|\d+h\d{2})",
            line,
        )

        if rest_match:
            current_block["rest"] = rest_match.group(1)

        # Block-level Credit.
        # Important: this should match "Credit 5h30M",
        # but NOT "Credit Time: 5h30D".
        credit_match = re.search(
            r"\bCredit\s+(?!Time:)(\d+h\d{2}[A-Z]?)",
            line,
        )

        if credit_match:
            current_block["credit"] = credit_match.group(1)

    trip["total_blocks"] = len(trip["blocks"])

    return trip


import pdfplumber


def extract_trips_from_pdf(
    pdf_path,
    first_page=2,
    last_page=None,
    stop_after_empty_pages=4,
    progress_callback=None,
):
    """
    Returns a dictionary keyed by Trip ID.

    progress_callback:
        Optional function that receives a progress dictionary.

    Example progress data:
        {
            "current": 10,
            "total": 150,
            "page": 11,
            "trips_on_page": 8,
            "total_trips": 72,
            "status": "running",
            "message": "Extracting page 11 of 150",
        }
    """

    def send_progress(
        current,
        total,
        page_number=None,
        trips_on_page=0,
        total_trips=0,
        status="running",
        message=None,
    ):
        if progress_callback is None:
            return

        progress_callback({
            "current": current,
            "total": total,
            "page": page_number,
            "trips_on_page": trips_on_page,
            "total_trips": total_trips,
            "status": status,
            "message": message,
        })

    trips = {}
    empty_pages_in_a_row = 0

    with pdfplumber.open(pdf_path) as pdf:
        total_pdf_pages = len(pdf.pages)

        start_index = first_page - 1
        end_index = last_page if last_page is not None else total_pdf_pages

        start_index = max(0, start_index)
        end_index = min(end_index, total_pdf_pages)

        total_pages_to_process = end_index - start_index

        if total_pages_to_process <= 0:
            send_progress(
                current=0,
                total=0,
                status="done",
                message="No pages to process.",
            )
            return trips

        send_progress(
            current=0,
            total=total_pages_to_process,
            status="starting",
            message="Starting trip extraction...",
        )

        for page_index in range(start_index, end_index):
            page_number = page_index + 1
            current_progress = page_index - start_index + 1

            page = pdf.pages[page_index]
            trips_found_on_page = 0

            try:
                page_words = page.extract_words(x_tolerance=1, y_tolerance=3) or []

                for bbox in make_trip_crops_from_words(page, page_words):
                    trip_words = words_inside_bbox(page_words, bbox)
                    text = words_to_text(trip_words)

                    if "Trip Id:" not in text:
                        continue

                    trip = parse_trip_text(text)
                    trips[trip["trip_id"]] = trip
                    trips_found_on_page += 1

            finally:
                page.close()

            if trips_found_on_page == 0:
                empty_pages_in_a_row += 1
            else:
                empty_pages_in_a_row = 0

            send_progress(
                current=current_progress,
                total=total_pages_to_process,
                page_number=page_number,
                trips_on_page=trips_found_on_page,
                total_trips=len(trips),
                status="running",
                message=(
                    f"Extracting trips: page {page_number} "
                    f"({current_progress} of {total_pages_to_process})"
                ),
            )

            if (
                stop_after_empty_pages is not None
                and empty_pages_in_a_row >= stop_after_empty_pages
            ):
                send_progress(
                    current=current_progress,
                    total=total_pages_to_process,
                    page_number=page_number,
                    trips_on_page=trips_found_on_page,
                    total_trips=len(trips),
                    status="stopped",
                    message=(
                        f"Stopped at page {page_number}: "
                        f"{empty_pages_in_a_row} empty pages in a row."
                    ),
                )
                break

        send_progress(
            current=min(current_progress, total_pages_to_process),
            total=total_pages_to_process,
            page_number=page_number,
            total_trips=len(trips),
            status="done",
            message=f"Finished extracting {len(trips)} trips.",
        )

    return trips

In [2]:
#Testing tool for speed
import time
import pdfplumber


def time_trip_extraction_by_page(pdf_path, first_page=2, last_page=None):
    with pdfplumber.open(pdf_path) as pdf:
        start_index = first_page - 1
        end_index = last_page if last_page is not None else len(pdf.pages)

        page_times = []

        for page_index in range(start_index, end_index):
            page_start = time.perf_counter()

            page = pdf.pages[page_index]
            trip_count = 0

            for bbox in make_trip_crops(page):
                cropped = page.crop(bbox)
                text = cropped.extract_text(x_tolerance=1, y_tolerance=3) or ""

                if "Trip Id:" not in text:
                    continue

                parse_trip_text(text)
                trip_count += 1

            elapsed = time.perf_counter() - page_start

            page_times.append({
                "page": page_index + 1,
                "trips": trip_count,
                "seconds": round(elapsed, 3),
            })

        return page_times

In [3]:
#Testing tool use

pdf_path = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Trips.pdf"
pdf_path = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/BID2605 trips 757 SDF.pdf"
page_times = time_trip_extraction_by_page(
    pdf_path,
    first_page=2,
)
total_time = 0
for item in page_times:
    print(item)
    total_time+= item['seconds']

print(total_time)
    

{'page': 2, 'trips': 6, 'seconds': 0.23}
{'page': 3, 'trips': 8, 'seconds': 0.223}
{'page': 4, 'trips': 8, 'seconds': 0.201}
{'page': 5, 'trips': 8, 'seconds': 0.198}
{'page': 6, 'trips': 8, 'seconds': 0.222}
{'page': 7, 'trips': 8, 'seconds': 0.19}
{'page': 8, 'trips': 8, 'seconds': 0.189}
{'page': 9, 'trips': 8, 'seconds': 0.193}
{'page': 10, 'trips': 8, 'seconds': 0.24}
{'page': 11, 'trips': 8, 'seconds': 0.198}
{'page': 12, 'trips': 8, 'seconds': 0.192}
{'page': 13, 'trips': 8, 'seconds': 0.258}
{'page': 14, 'trips': 8, 'seconds': 0.192}
{'page': 15, 'trips': 8, 'seconds': 0.193}
{'page': 16, 'trips': 8, 'seconds': 0.191}
{'page': 17, 'trips': 8, 'seconds': 0.276}
{'page': 18, 'trips': 8, 'seconds': 0.19}
{'page': 19, 'trips': 8, 'seconds': 0.189}
{'page': 20, 'trips': 8, 'seconds': 0.198}
{'page': 21, 'trips': 8, 'seconds': 0.192}
{'page': 22, 'trips': 8, 'seconds': 0.301}
{'page': 23, 'trips': 8, 'seconds': 0.195}
{'page': 24, 'trips': 8, 'seconds': 0.196}
{'page': 25, 'trips': 8

In [ ]:
from pprint import pprint
pprint(trips)

In [ ]:
# do not know what this is 
def parse_trip_text(text):
    """
    Parses one trip table into a dictionary.

    Extracts:
        Trip-level:
            - trip_id
            - lines
            - total_blocks
            - tafb
            - premium
            - duty_time
            - block_time
            - credit_time
            - per_diem
            - ldgs

        Block-level:
            - start
            - end
            - duty
            - block
            - rest
            - credit
            - flights
    """

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    full_text = "\n".join(lines)

    trip_id = int(re.search(r"Trip Id:\s*(\d+)", full_text).group(1))

    lines_match = re.search(r"Lines:\s*([^\n]+)", full_text)
    line_numbers = []

    if lines_match:
        line_numbers = [int(x) for x in re.findall(r"\d+", lines_match.group(1))]

    trip = {
        "trip_id": trip_id,
        "lines": line_numbers,
        "total_blocks": 0,

        # Trip summary values
        "tafb": get_first_match(r"TAFB:\s*(\d+h\d{2})", full_text),
        "premium": get_first_float(r"Premium\s+([\d.]+)", full_text),
        "duty_time": get_first_match(r"Duty Time:\s*(\d+h\d{2})", full_text),
        "block_time": get_first_match(r"Block Time:\s*(\d+h\d{2})", full_text),
        "credit_time": get_first_match(r"Credit Time:\s*(\d+h\d{2}[A-Z]?)", full_text),
        "per_diem": get_first_float(r"per Diem\s+([\d.]+)", full_text),
        "ldgs": get_first_int(r"LDGS\s+(\d+)", full_text),

        # Block list
        "blocks": [],
    }

    current_block = None

    for line in lines:
        # New block starts with something like:
        # (15)19:43 1h00 Duty 8h26
        # (16)21:00 0h00 Duty 0h00
        duty_match = re.match(
            rf"^(?P<block_start>{TIME_RE})\s+"
            rf"(?P<report_time>{DUR_RE})\s+"
            rf"Duty\s+(?P<duty>{DUR_RE})",
            line,
        )

        if duty_match:
            current_block = {
                "start": clean_time(duty_match.group("block_start")),
                "end": None,

                # New block-level fields
                "duty": duty_match.group("duty"),
                "block": None,
                "rest": None,
                "credit": None,

                "flights": [],
            }
            trip["blocks"].append(current_block)
            continue

        if current_block is None:
            continue

        # Block ends can look like:
        # (00)04:09 0h15 Credit 4h13D
        # (03)07:00 0h00 Rest -
        # (20)00:51 0h15 Credit 4h00M
        #
        # But this should NOT count as a block end:
        # (16)21:00 0h00 Duty 0h00
        block_end_match = re.match(
            rf"^(?P<block_end>{TIME_RE})\s+"
            rf"(?P<ground_time>{DUR_RE})"
            rf"(?:\s+(?P<after>.*))?$",
            line,
        )

        if block_end_match:
            after = block_end_match.group("after") or ""

            if not after.startswith("Duty"):
                current_block["end"] = clean_time(block_end_match.group("block_end"))

        flight = parse_flight_line(line)

        if flight:
            # For the first flight in each block, use the duty/block start time,
            # not the actual flight departure time.
            if not current_block["flights"]:
                flight["start"] = current_block["start"]

            current_block["flights"].append(flight)

        # Block-level Block.
        # Important: this should match "Block 3h19",
        # but NOT "Block Time: 3h19".
        block_match = re.search(
            r"\bBlock\s+(?!Time:)(\d+h\d{2})",
            line,
        )

        if block_match:
            current_block["block"] = block_match.group(1)

        # Block-level Rest.
        rest_match = re.search(
            r"\bRest\s+(-|\d+h\d{2})",
            line,
        )

        if rest_match:
            current_block["rest"] = rest_match.group(1)

        # Block-level Credit.
        # Important: this should match "Credit 5h30M",
        # but NOT "Credit Time: 5h30D".
        credit_match = re.search(
            r"\bCredit\s+(?!Time:)(\d+h\d{2}[A-Z]?)",
            line,
        )

        if credit_match:
            current_block["credit"] = credit_match.group(1)

    trip["total_blocks"] = len(trip["blocks"])

    return trip

In [ ]:
#Multithreading and Multiprocessing attempt
import re
import pdfplumber


TIME_RE = r"\(\d{2}\)\d{2}:\d{2}"
DUR_RE = r"\d+h\d{2}"

def clean_time(value):
    """
    Converts:
        (16)20:43 -> 20:43
        (00)04:09 -> 04:09
    """
    if value is None:
        return None

    return re.sub(r"^\(\d{2}\)", "", value)


def group_words_by_line(words, tolerance=2):
    lines = []

    for word in sorted(words, key=lambda w: (w["top"], w["x0"])):
        for line in lines:
            if abs(line[0]["top"] - word["top"]) <= tolerance:
                line.append(word)
                break
        else:
            lines.append([word])

    return [sorted(line, key=lambda w: w["x0"]) for line in lines]


def find_trip_anchors(page):
    """
    Finds each 'Trip Id: ###' on the page.
    Needed because each page can contain several trip tables.
    """
    words = page.extract_words(x_tolerance=1, y_tolerance=3) or []
    anchors = []

    for line in group_words_by_line(words):
        text_parts = [w["text"] for w in line]

        for i in range(len(text_parts) - 2):
            if (
                text_parts[i] == "Trip"
                and text_parts[i + 1] == "Id:"
                and text_parts[i + 2].isdigit()
            ):
                anchors.append({
                    "trip_id": int(text_parts[i + 2]),
                    "x0": line[i]["x0"],
                    "top": line[i]["top"],
                })

    return sorted(anchors, key=lambda a: (a["top"], a["x0"]))


def make_trip_crops(page):
    """
    Builds a crop box for each trip table.

    The crop is necessary because the PDF pages can have left/right tables.
    Without cropping, pdfplumber may mix text from different tables.
    """
    anchors = find_trip_anchors(page)
    page_middle = page.width / 2

    for anchor in anchors:
        anchor["column"] = 0 if anchor["x0"] < page_middle else 1

    crops = []

    for anchor in anchors:
        next_trip_same_column = [
            other
            for other in anchors
            if other["column"] == anchor["column"]
            and other["top"] > anchor["top"] + 5
        ]

        bottom = min(
            [other["top"] for other in next_trip_same_column],
            default=page.height - 10,
        )

        if anchor["column"] == 0:
            x0, x1 = 0, page_middle - 2
        else:
            x0, x1 = page_middle + 2, page.width

        crops.append((
            x0,
            max(0, anchor["top"] - 3),
            x1,
            min(page.height, bottom - 2),
        ))

    return crops


def split_route(route_raw):
    """
    Examples:
        SDF-PHL         -> departure SDF, arrival PHL, route_flags []
        SDF-IRO-PHL     -> departure SDF, arrival PHL, route_flags ['IRO']
        SDF-BDL(C)      -> departure SDF, arrival BDL, route_flags ['C']
        SDF-IRO-BDL(C)  -> departure SDF, arrival BDL, route_flags ['IRO', 'C']
    """
    parts = route_raw.split("-")

    airports = []
    route_flags = []

    for part in parts:
        if part == "IRO":
            route_flags.append("IRO")
            continue

        # Handles airport with parenthetical flag, like BDL(C)
        match = re.match(r"^([A-Z]{3})(?:\(([A-Z]+)\))?$", part)

        if match:
            airport = match.group(1)
            flag = match.group(2)

            airports.append(airport)

            if flag:
                route_flags.append(flag)
        else:
            airports.append(part)

    departure = airports[0] if airports else None
    arrival = airports[-1] if airports else None

    return departure, arrival, route_flags


def parse_flight_line(line):
    """
    Extracts only:
    - flight
    - route_raw
    - departure
    - arrival
    - route_flags, such as IRO
    - start
    - end
    """

    match = re.match(r"^\d+\s+\([^)]*\)[A-Za-z]{0,2}\s+(.*)$", line)

    if not match:
        return None

    body = match.group(1)

    route_match = re.search(
        r"[A-Z]{3}(?:\([A-Z]\))?(?:-(?:IRO|[A-Z]{3}(?:\([A-Z]\))?))+",
        body,
    )

    if not route_match:
        return None

    flight = body[:route_match.start()].strip()
    route_raw = route_match.group(0)
    after_route = body[route_match.end():].strip()

    time_match = re.match(
        rf"(?P<start>{TIME_RE})\s+(?P<end>{TIME_RE})",
        after_route,
    )

    if not time_match:
        return None

    departure, arrival, route_flags = split_route(route_raw)

    return {
        "flight": flight,
        "route_raw": route_raw,
        "departure": departure,
        "arrival": arrival,
        "route_flags": route_flags,
        "start": clean_time(time_match.group("start")),
        "end": clean_time(time_match.group("end")),
    }


def parse_trip_text(text):
    """
    Parses one trip table into a dictionary.
    """

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    full_text = "\n".join(lines)

    trip_id = int(re.search(r"Trip Id:\s*(\d+)", full_text).group(1))

    lines_match = re.search(r"Lines:\s*([^\n]+)", full_text)
    line_numbers = []

    if lines_match:
        line_numbers = [int(x) for x in re.findall(r"\d+", lines_match.group(1))]

    tafb_match = re.search(r"TAFB:\s*(\d+h\d{2})", full_text)
    premium_match = re.search(r"Premium\s+([\d.]+)", full_text)

    trip = {
        "trip_id": trip_id,
        "lines": line_numbers,
        "total_blocks": 0,
        "tafb": tafb_match.group(1) if tafb_match else None,
        "premium": float(premium_match.group(1)) if premium_match else None,
        "blocks": [],
    }

    current_block = None

    for line in lines:
        # New block starts with something like:
        # (15)19:43 1h00 Duty 8h26
        duty_match = re.match(
            rf"^(?P<block_start>{TIME_RE})\s+{DUR_RE}\s+Duty\s+{DUR_RE}",
            line,
        )

        if duty_match:
            current_block = {
                "start": clean_time(duty_match.group("block_start")),
                "end": None,
                "rest": None,
                "flights": [],
            }
            trip["blocks"].append(current_block)
            continue

        if current_block is None:
            continue

        # Block ends with something like:
        # (00)04:09 0h15 Credit 4h13D
        block_end_match = re.match(
            rf"^(?P<block_end>{TIME_RE})\s+"
            rf"(?P<ground_time>{DUR_RE})"
            rf"(?:\s+(?P<after>.*))?$",
            line,
        )

        if block_end_match:
            after = block_end_match.group("after") or ""

            # If the word immediately after the duration is Duty,
            # this is a new block start, not a block end.
            if not after.startswith("Duty"):
                current_block["end"] = clean_time(block_end_match.group("block_end"))

        flight = parse_flight_line(line)

        if flight:
            # For the first flight in each block, use the block/duty start time,
            # not the actual flight departure time.
            if not current_block["flights"]:
                flight["start"] = current_block["start"]

            current_block["flights"].append(flight)

        rest_match = re.search(r"Rest\s+(-|\d+h\d{2})", line)

        if rest_match:
            current_block["rest"] = rest_match.group(1)

    trip["total_blocks"] = len(trip["blocks"])

    return trip


def extract_trips_from_pdf(pdf_path, first_page=2, last_page=None):
    """
    Returns a dictionary keyed by Trip ID.

    Page numbers are normal PDF page numbers:
        first_page=2 means page 2.
        last_page=4 means page 4.
    """

    trips = {}

    with pdfplumber.open(pdf_path) as pdf:
        start_index = first_page - 1
        end_index = last_page if last_page is not None else len(pdf.pages)

        for page_index in range(start_index, end_index):
            page = pdf.pages[page_index]

            for bbox in make_trip_crops(page):
                cropped = page.crop(bbox)
                text = cropped.extract_text(x_tolerance=1, y_tolerance=3) or ""

                if "Trip Id:" not in text:
                    continue

                trip = parse_trip_text(text)
                trips[trip["trip_id"]] = trip

    return trips

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from concurrent.futures.process import BrokenProcessPool
from math import ceil
import os
import pdfplumber


def get_auto_worker_count(page_count=None, reserve_cores=1, max_limit=8):
    cpu_count = os.cpu_count() or 1

    workers = max(1, cpu_count - reserve_cores)

    if max_limit is not None:
        workers = min(workers, max_limit)

    if page_count is not None:
        workers = min(workers, page_count)

    return max(1, workers)


def _make_page_ranges(start_index, end_index, max_workers):
    total_pages = end_index - start_index

    if total_pages <= 0:
        return []

    max_workers = max(1, min(max_workers, total_pages))
    chunk_size = ceil(total_pages / max_workers)

    ranges = []

    for start in range(start_index, end_index, chunk_size):
        stop = min(start + chunk_size, end_index)
        ranges.append((start, stop))

    return ranges


def _extract_trips_from_page_range(args):
    pdf_path, start_index, end_index = args

    trips = {}

    with pdfplumber.open(pdf_path) as pdf:
        for page_index in range(start_index, end_index):
            page = pdf.pages[page_index]

            for bbox in make_trip_crops(page):
                cropped = page.crop(bbox)
                text = cropped.extract_text(x_tolerance=1, y_tolerance=3) or ""

                if "Trip Id:" not in text:
                    continue

                try:
                    trip = parse_trip_text(text)
                    trips[trip["trip_id"]] = trip

                except Exception as e:
                    print(
                        f"Could not parse trip on PDF page {page_index + 1}: {e}"
                    )

    return trips


def _run_trip_workers(worker_args, max_workers, executor_class):
    trips = {}

    with executor_class(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_extract_trips_from_page_range, args)
            for args in worker_args
        ]

        for future in as_completed(futures):
            trips.update(future.result())

    return trips


def extract_trips_from_pdf_parallel(
    pdf_path,
    first_page=2,
    last_page=None,
    max_workers=None,
    use_processes=True,
    fallback_to_threads=True,
):
    """
    Parallel trip extraction.

    use_processes=True:
        Usually faster in a normal .py script.

    use_processes=False:
        Safer in Jupyter/notebooks and packaged apps.
    """

    with pdfplumber.open(pdf_path) as pdf:
        total_pdf_pages = len(pdf.pages)

    start_index = first_page - 1
    end_index = last_page if last_page is not None else total_pdf_pages

    start_index = max(0, start_index)
    end_index = min(end_index, total_pdf_pages)

    if end_index <= start_index:
        return {}

    page_count = end_index - start_index

    if max_workers is None:
        max_workers = get_auto_worker_count(
            page_count=page_count,
            reserve_cores=1,
            max_limit=8,
        )

    max_workers = max(1, min(max_workers, page_count))

    page_ranges = _make_page_ranges(
        start_index=start_index,
        end_index=end_index,
        max_workers=max_workers,
    )

    worker_args = [
        (pdf_path, range_start, range_end)
        for range_start, range_end in page_ranges
    ]

    if use_processes:
        try:
            return _run_trip_workers(
                worker_args=worker_args,
                max_workers=max_workers,
                executor_class=ProcessPoolExecutor,
            )

        except BrokenProcessPool:
            if not fallback_to_threads:
                raise

            print("Process pool crashed. Retrying with threads...")

            return _run_trip_workers(
                worker_args=worker_args,
                max_workers=max_workers,
                executor_class=ThreadPoolExecutor,
            )

    return _run_trip_workers(
        worker_args=worker_args,
        max_workers=max_workers,
        executor_class=ThreadPoolExecutor,
    )

In [ ]:
#Testing tool
import time
import pdfplumber


def time_trip_extraction_by_page(pdf_path, first_page=2, last_page=None):
    with pdfplumber.open(pdf_path) as pdf:
        start_index = first_page - 1
        end_index = last_page if last_page is not None else len(pdf.pages)

        page_times = []

        for page_index in range(start_index, end_index):
            page_start = time.perf_counter()

            page = pdf.pages[page_index]
            trip_count = 0

            for bbox in make_trip_crops(page):
                cropped = page.crop(bbox)
                text = cropped.extract_text(x_tolerance=1, y_tolerance=3) or ""

                if "Trip Id:" not in text:
                    continue

                parse_trip_text(text)
                trip_count += 1

            elapsed = time.perf_counter() - page_start

            page_times.append({
                "page": page_index + 1,
                "trips": trip_count,
                "seconds": round(elapsed, 3),
            })

        return page_times

In [ ]:
#Testing tool use
page_times = time_trip_extraction_by_page(
    pdf_path,
    first_page=2,
)
total_time = 0
for item in page_times:
    print(item)
    total_time+= item['seconds']

print(total_time)
    

In [ ]:
from pprint import pprint

pdf_path = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Trips.pdf"

trips = extract_trips_from_pdf_parallel(pdf_path,first_page=2,use_processes=True)

pprint(trips[607])
